# Lab 4.3 &mdash; Your Own Repos, Your Own Identity

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Add a third MCP server to the agent from Labs 4.1 and 4.2
- Authenticate as <em>yourself</em> for the first time in this module
- Get review, release-note and triage work done against your own repositories
- Leave with prompts you can point at the repo you work in on Monday

> **How this lab works &mdash; it is different from the others.** There is nothing to fill in
> and nothing to score. You run the cells in order and watch a real agent reach a real Jira
> over MCP. The participant notebook and the solution notebook are the same file, on purpose:
> the point is to *see the protocol work* before Module 4 asks you to build one. Read the
> output of every cell &mdash; that is the lab.

> **Nothing to fill in**, but this one needs something from you: a GitHub account and a
> read-only token. Five minutes, and it is the only lab where the credential is yours.

## The use case

You already know the shape. What changes here is **whose identity the agent is using.**

| | credential | who GitHub/Jira/Langfuse thinks is acting |
|---|---|---|
| Lab 4.1 &mdash; Jira | one we issued | a shared service account &mdash; the audit trail says the same name for all thirty of you |
| Lab 4.2 &mdash; Langfuse | injected by the sandbox | one shared project, separated only by an environment tag |
| **Lab 4.3 &mdash; GitHub** | **yours** | **you** |

That is not a detail. Every action the agent takes here is attributable to you, appears in your
contribution history, and is bounded by what *your* token is allowed to do. It is the first time
in Module 4 that the answer to *&ldquo;who did that?&rdquo;* is a person.

Which is also why this is the lab where the token scope matters.

## Step 0 &mdash; Make a read-only token (about three minutes)

**Do this before running anything.**

1. Go to **github.com &rarr; Settings &rarr; Developer settings &rarr; Personal access tokens &rarr;
   Fine-grained tokens &rarr; Generate new token**
2. **Expiration:** 7 days. This is a workshop.
3. **Repository access:** *Only select repositories* &mdash; pick one or two you actually work in.
   Public repositories work fine too if you would rather not point it at anything real.
4. **Permissions &rarr; Repository permissions**, set these to **Read-only** and nothing else:
   `Contents`, `Issues`, `Pull requests`, `Metadata`
5. Generate, and copy it. You will paste it once, in a moment, and it is never written to disk by
   this notebook.

> **Why read-only.** The server you are about to connect publishes `delete_file`,
> `merge_pull_request` and `create_repository`. Step 3 refuses those at the client, but a token
> that cannot do them in the first place is a stronger control than a config file that declines to
> ask. Least privilege at the credential beats least privilege at the client, every time.

In [ ]:
# ------------------------------------------------------------ Preflight: run me first
import os, sys, json, time, getpass, textwrap, subprocess, re, socket, urllib.request, urllib.error

GH_MCP = "https://api.githubcopilot.com/mcp/"
LABDIR = os.path.expanduser("~/work/mcplab")          # the SAME folder as Labs 4.1 and 4.2
CONFIG_PATH = os.path.join(LABDIR, "opencode.json")


def ask(prompt: str, secret: bool = False) -> str:
    """Prompt the participant, but never block a headless run (the lab verifiers)."""
    try:
        if not sys.stdin.isatty() and "ipykernel" not in sys.modules:
            return ""
        return (getpass.getpass(prompt) if secret else input(prompt)).strip()
    except Exception:
        return ""          # nbclient runs with stdin disabled; that is fine, we just skip


GH_USER  = ask("Your GitHub username: ")
GH_TOKEN = ask("Paste your fine-grained token (input is hidden, not saved): ", secret=True)

print()
print("lab folder     :", LABDIR, "-", "found" if os.path.exists(CONFIG_PATH) else "will be created")
print("github user    :", GH_USER or "(not entered - later cells will skip)")
print("token          :", f"received, {len(GH_TOKEN)} chars, held in memory only" if GH_TOKEN else "(not entered)")

def ready() -> bool:
    return bool(GH_USER and GH_TOKEN)

## Step 1 &mdash; Prove the token is yours before trusting it

A username you typed and a token you pasted are two independent claims. `get_me` settles both: it
returns whoever GitHub thinks is holding that token.

If the login it returns is not the name you typed, you have pasted the wrong token &mdash; better to
find out now than three prompts later when the agent quietly acts as somebody else.

In [ ]:
def gh_rpc(method, params=None, sid=None, timeout=90):
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}).encode()
    req = urllib.request.Request(GH_MCP, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("Accept", "application/json, text/event-stream")
    req.add_header("Authorization", "Bearer " + GH_TOKEN)
    if sid:
        req.add_header("Mcp-Session-Id", sid)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw, sess = r.read().decode(), r.headers.get("mcp-session-id")
    for line in raw.splitlines():
        if line.startswith("data:"):
            raw = line[5:].strip()
            break
    return json.loads(raw).get("result", {}), sess


if ready():
    try:
        init, gh_session = gh_rpc("initialize", {
            "protocolVersion": "2025-06-18", "capabilities": {},
            "clientInfo": {"name": "lab-4-3", "version": "1.0"}})
        print("server :", init.get("serverInfo", {}).get("name"))

        res, _ = gh_rpc("tools/call", {"name": "get_me", "arguments": {}}, sid=gh_session)
        me = json.loads("".join(c.get("text", "") for c in res.get("content", [])))
        login = me.get("login")

        print("token belongs to:", login)
        if login and GH_USER and login.lower() != GH_USER.lower():
            print(f"\n  !! you typed '{GH_USER}' but the token belongs to '{login}'.")
            print("     Use the login above in the prompts, or paste the right token.")
        else:
            print("matches what you typed. Good.")
    except urllib.error.HTTPError as e:
        print(f"GitHub refused the token (HTTP {e.code}). Check it was copied whole and has not expired.")
else:
    print("skipped - no username/token entered in the preflight cell")

## Step 2 &mdash; Add it to the agent you already have

Third entry in the same `mcp` block. Jira, Langfuse, GitHub &mdash; one agent, three systems, three
separate credentials, each doing its own job.

Note what does **not** go in the file: the token. It is referenced as `{env:GITHUB_PAT}` and read
from the environment at run time, exactly as in Lab 4.1. That is what keeps this config safe to
commit, share, or paste into a ticket.

In [ ]:
def load_config() -> dict:
    if os.path.exists(CONFIG_PATH):
        return json.load(open(CONFIG_PATH))
    return {
        "$schema": "https://opencode.ai/config.json",
        "provider": {"litellm": {
            "npm": "@ai-sdk/openai-compatible", "name": "LiteLLM Gateway",
            "options": {"baseURL": "{env:LAB_LLM_BASE_URL}", "apiKey": "{env:LITELLM_API_KEY}"},
            "models": {"qwen36-35b-a3b-lab": {"name": "Qwen3.6 35B A3B (lab)"}}}},
        "mcp": {},
    }


if ready():
    os.makedirs(LABDIR, exist_ok=True)
    cfg = load_config()
    before = sorted(cfg.get("mcp", {}))

    cfg.setdefault("mcp", {})["github"] = {
        "type": "remote",
        "url": GH_MCP,
        "enabled": True,
        "headers": {"Authorization": "Bearer {env:GITHUB_PAT}"},   # the token stays out of the file
    }

    # Defence in depth. Your read-only token already forbids these; saying so here means the
    # agent is never even offered them, so it cannot try and cannot be talked into trying.
    cfg.setdefault("permission", {}).update({
        "github_delete*":            "deny",
        "github_merge*":             "deny",
        "github_create_repository":  "deny",
        "github_push*":              "ask",
        "github_create_or_update*":  "ask",
    })

    with open(CONFIG_PATH, "w") as fh:
        json.dump(cfg, fh, indent=2)

    print("servers before :", before or "(none - you skipped 4.1 and 4.2, that is fine)")
    print("servers after  :", sorted(cfg["mcp"]))
    print("\nwrote", CONFIG_PATH)
else:
    print("skipped - see the preflight cell")

## Step 3 &mdash; Export the token and confirm

`opencode` reads `GITHUB_PAT` from the environment of the terminal it runs in, so export it there.
Paste the token again when you do &mdash; the notebook deliberately never wrote it anywhere.

In [ ]:
if ready():
    print("In your terminal (File > New > Terminal):\n")
    print(f"  cd {LABDIR}")
    print( "  export GITHUB_PAT=<paste your token>")
    print( "  opencode mcp list\n")
    print("You should see three servers, all connected: jira, langfuse, github.")
    print("\nIf github says 'needs authentication', GITHUB_PAT is unset or empty in that shell -")
    print("the same failure mode as Lab 4.1, and the same fix.")
else:
    print("skipped - see the preflight cell")

## The prompt library

Point these at a repository you actually work in. Replace `<repo>` with `owner/name` &mdash; the
agent needs the full form. Everything here works with a **read-only** token.

**Two ground rules:**

1. **Name the repo explicitly.** The server has no idea which of your repositories you mean, and
   guessing costs it a search.
2. ⚠️ **Read the answer as a draft, not a verdict.** These prompts summarise and prioritise; both
   are judgements. The agent is quoting real commits and real issues, but the ranking is its
   opinion.

### The Monday morning questions

```
Summarise what changed in <repo> over the last 7 days: which files moved, what the commits
were about, and what I should look at first if I have only twenty minutes.
```
*Otherwise: scrolling the commit list and guessing. This is the one to run before standup.*

```
List the open pull requests in <repo> with how long each has been waiting, who is blocking
it, and which are safe to merge on the strength of their description and checks.
```
*Review queues rot because nobody sorts them. This sorts them.*

```
Find the open issues in <repo> that have no assignee, group them by the area of the codebase
they touch, and tell me which look like quick wins.
```
*Backlog triage that otherwise happens once a quarter, badly.*

### Understanding code you did not write

```
In <repo>, find where <FunctionOrClass> is defined and every place it is used. Summarise
what it does and what would break if I changed its signature.
```
*The question you ask on day one in a new codebase, and again every time you touch something
unfamiliar.*

```
Read the last 30 commits on <repo> and write release notes grouped into Features, Fixes and
Internal. Plain text, no markdown headings.
```
*Release notes nobody wants to write, from the data that already describes them.*

```
Explain the purpose of <repo> from its README, its top-level layout and its most recently
changed files. Assume I am joining the team tomorrow.
```
*Onboarding, compressed.*

### Reviewing, and filing well

```
Look at pull request #<N> in <repo>. Summarise what it changes, then list what you would
question in review — correctness, missing tests, anything that looks unrelated to the
stated purpose.
```
*A first-pass review before you spend your own attention. Treat it as a checklist, not a verdict.*

```
I want to raise this in <repo>: "<your rough note>". FIRST search the existing open and
closed issues for anything covering the same thing and tell me what you found. Only if
nothing matches, draft the issue text — do not create it yet. Plain text, no markdown.
```
*The same dedup-before-filing discipline as Lab 4.1, and the same reason: the check humans skip
is the one that costs the team.*

### One worth running to see it refuse

```
Delete the README from <repo>.
```
*Two independent controls say no: your read-only token cannot, and Step 2 denied `github_delete*`
so the agent is never offered the tool. Read which one it reports &mdash; and notice that the
config-level refusal is the one that happens without a single network call.*

## What this bought you, and what to take away

Three labs, three servers, one agent, and one config file that grew by four lines each time.

**The thing worth remembering is the credential, not the protocol.** MCP made all three
integrations look identical &mdash; a URL, a header, a tool list. What differs is entirely in who
the header says you are:

- a **shared service account** (4.1): convenient, and the audit log is useless
- an **injected project key** (4.2): everyone's data in one place, separated by a convention
- **your own scoped token** (4.3): attributable, revocable, and bounded by what you granted it

When someone asks whether it is safe to give an agent access to a system, that is the question they
are actually asking. The answer is never about MCP. It is about which of those three you handed it,
and how narrow you were willing to make it.

## Your turn

- Point the review prompt at a PR you already reviewed by hand. Compare. Where was it useful, and
  where would trusting it have cost you?
- Re-run `opencode mcp list` and count the tools across all three servers. That total is what your
  agent carries into every turn.
- Revoke the token when the workshop ends. github.com &rarr; Settings &rarr; Developer settings.
  It expires in 7 days anyway &mdash; do it deliberately, and notice how quickly the access you
  granted disappears.